# MULDE Training Pipeline
This notebook clones your modified MULDE repository, mounts Google Drive to access the Hiera-L features, and trains the MULDE anomaly detection model.

**Features:**
- Checkpoints saved to Google Drive every 50 epochs (survives Colab crashes)
- Automatically resumes from the last checkpoint
- AUC-ROC scores printed directly to terminal (no TensorBoard needed)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
repo_dir = '/content/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection'

if os.path.exists(repo_dir):
    print('Repository already cloned. Pulling latest changes...')
    !cd {repo_dir} && git pull
else:
    !git clone -b First-Branch https://github.com/Hadi6618/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection.git

%cd {repo_dir}

## Setup Ground Truth Labels
The test phase requires the frame-level ground truth `.npy` labels for the ShanghaiTech dataset.

Modify the path below if your labels are stored in a different location.

In [ ]:
# Uncomment and modify if you need to extract the labels from a zip on Drive:
# !unzip -q "/content/drive/MyDrive/path_to_shanghaitech.zip" -d "/content/shanghaitech"

import os
label_dir = '/content/shanghaitech/testing/label'
if os.path.isdir(label_dir):
    n_labels = len([f for f in os.listdir(label_dir) if f.endswith('.npy')])
    print(f'Found {n_labels} label files in {label_dir}')
else:
    print(f'WARNING: Label directory not found at {label_dir}')
    print('Please extract or mount the ShanghaiTech labels before training.')

## Run MULDE Training
Parameters match the CVPR 2024 paper for ShanghaiTech.

- **Checkpoints** saved to Google Drive every 50 epochs
- **Evaluation** runs every 50 epochs (and at the final epoch)
- **Auto-resume**: If Colab crashes, just re-run this cell and it picks up where it left off

In [ ]:
!python main.py \
  --experiment_name "HieraL_ShanghaiTech" \
  --train_dir "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train" \
  --test_dir "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/test" \
  --test_labels_dir "/content/shanghaitech/testing/label" \
  --stats_path "/content/drive/MyDrive/MULDE/features/hiera_large_16x224_mae_k400_ft_k400_centered_s4/train_feature_stats.npz" \
  --checkpoint_dir "/content/drive/MyDrive/MULDE/checkpoints" \
  --batch_size 2048 \
  --epochs 1000 \
  --lr 4e-5 \
  --beta 0.1 \
  --L 16 \
  --sigma_low 1e-3 \
  --sigma_high 1. \
  --units 4096 4096 \
  --eval_freq 50 \
  --checkpoint_freq 50